In [0]:
from pyspark.sql import functions as F
from config import load_config

cfg = load_config("config_databricks.yaml")

candidates = (
    spark.table(cfg["bronze_candidates_table"])
    .select(
        F.trim(F.col("candidate_id")).alias("candidate_id"),
        F.trim(F.col("first_name")).alias("first_name"),
        F.trim(F.col("last_name")).alias("last_name"),
        F.trim(F.col("email")).alias("email"),
        F.trim(F.col("phone")).alias("phone"),
        F.trim(F.col("skills")).alias("skills")
    )
    .dropDuplicates(["candidate_id"])
)

education = (
    spark.table(cfg["bronze_education_table"])
    .groupBy("candidate_id")
    .agg(
        F.collect_list(
            F.struct(
                F.trim(F.col("degree")).alias("degree"),
                F.trim(F.col("institution")).alias("institution"),
                F.trim(F.col("year")).alias("year")
            )
        ).alias("education")
    )
)

dim_candidate = candidates.join(education, on="candidate_id", how="left")
dim_candidate.write.format("delta").mode("overwrite").saveAsTable(cfg["dim_candidate_table"])

In [0]:
spark.sql("SELECT COUNT(*) FROM dim_candidate").show()
spark.sql("SELECT * FROM dim_candidate WHERE education IS NULL LIMIT 10").show()
spark.sql("SELECT * FROM dim_candidate WHERE education IS NOT NULL LIMIT 10").show()
spark.sql("SELECT COUNT(*) as count, candidate_id FROM dim_candidate GROUP BY candidate_id HAVING count > 1 LIMIT 10").show()